# 02. 데이터 품질 + 텍스트 분석

**목표**: 데이터 품질 이슈 발견 + 텍스트 길이 분포로 청킹 전략 도출

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase3_quality.json`, `eda_output/phase4_text.json`

In [1]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import re
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    get_sample,
    stream_json,
    load_result,
    save_result,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

# ┌──────────────────────────────────────────────────────────┐
# │  분석 모드 선택                                          │
# │                                                          │
# │  True  = 전체 데이터 스트리밍 순회 (메모리 절약)         │
# │  False = 샘플 데이터 (카테고리당 5,000건)                │
# └──────────────────────────────────────────────────────────┘
USE_FULL_DATA = True

# 이전 단계 결과 로드
phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
print(f"Phase 1 로드: {len(phase1)}개 파일")
print(f"Phase 2 로드: {len(phase2)}개 카테고리 스키마")

# ── 데이터 로딩 메타 구성 (레코드 캐시 대신 스트리밍 헬퍼) ───
SAMPLE_SIZE = 5000
_available_categories: set[str] = set()


def iter_category_records(cat_key: str):
    """카테고리 레코드를 스트리밍/샘플 방식으로 순회."""
    cat_info = CATEGORIES[cat_key]
    for fname in cat_info["files"]:
        filepath = DATA_DIR / fname
        if not filepath.exists():
            continue
        if USE_FULL_DATA:
            yield from stream_json(filepath)
        else:
            for rec in get_sample(filepath, n=SAMPLE_SIZE, fast=True):
                yield rec


mode = "전체 데이터 (스트리밍)" if USE_FULL_DATA else f"샘플 데이터 ({SAMPLE_SIZE:,}건)"
print(f"\n분석 모드: {mode}")

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="카테고리 스캔"):
    existing = 0
    est_records = 0
    for fname in cat_info["files"]:
        filepath = DATA_DIR / fname
        if filepath.exists():
            existing += 1
    if existing == 0:
        continue
    est_records = sum(
        f["record_count"]
        for f in phase1
        if f.get("category") == cat_key
    )
    _available_categories.add(cat_key)
    file_count = len(cat_info["files"])
    file_note = f" ({file_count}개 파일)" if file_count > 1 else ""
    print(f"  {cat_info['label']}{file_note}: 약 {est_records:,}건")

print(f"\n총 {len(_available_categories)}개 카테고리 준비 완료")




/Users/gimjuhyeong/dev/law-3-team/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Phase 1 로드: 48개 파일
Phase 2 로드: 11개 카테고리 스키마

분석 모드: 전체 데이터 (스트리밍)


카테고리 스캔: 100%|██████████| 11/11 [00:00<00:00, 11437.12it/s]

  판례: 약 92,055건
  법령: 약 5,548건
  헌재결정례: 약 31,718건
  행정심판례: 약 34,254건
  특별행정심판 (2개 파일): 약 148,778건
  법령해석례: 약 8,597건
  위원회 결정문 (10개 파일): 약 56,802건
  부처 해석례 (28개 파일): 약 37,325건
  법률용어사전: 약 81,488건
  조약: 약 3,589건
  행정규칙: 약 5,258건

총 11개 카테고리 준비 완료


## 1. 데이터 품질 분석

카테고리별 5,000건 샘플에서 null/empty 비율, 중복 ID, 인코딩 이슈를 점검합니다.

In [2]:
# ── 카테고리별 품질 분석 ──────────────────────────────────
quality_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="품질 분석"):
    if cat_key not in _available_categories:
        continue

    total = 0
    field_quality: dict[str, dict] = {}

    id_field = cat_info.get("id_field")
    id_counts: Counter = Counter()

    encoding_issues = 0
    html_entities = 0

    for record in iter_category_records(cat_key):
        total += 1

        for key, value in record.items():
            if key not in field_quality:
                field_quality[key] = {"null": 0, "empty": 0, "present": 0, "null_str": 0}
            fq = field_quality[key]
            fq["present"] += 1
            if value is None:
                fq["null"] += 1
            elif isinstance(value, str):
                if value.strip() == "":
                    fq["empty"] += 1
                elif value.strip().lower() == "null":
                    fq["null_str"] += 1

        if id_field:
            v = record.get(id_field)
            if v is not None:
                id_counts.update([str(v) if isinstance(v, list) else v])

        if total <= 1000:
            for value in record.values():
                if not isinstance(value, str):
                    continue
                if "\ufffd" in value or "\\u" in value:
                    encoding_issues += 1
                if "&amp;" in value or "&lt;" in value or "&gt;" in value:
                    html_entities += 1

    if total == 0:
        continue

    duplicate_ids = sum(1 for c in id_counts.values() if c > 1)
    duplicate_rate = duplicate_ids / len(id_counts) if id_counts else 0.0

    quality_results[cat_key] = {
        "label": cat_info["label"],
        "sample_count": total,
        "field_quality": {
            k: {
                "null_rate": round(v["null"] / v["present"], 4) if v["present"] > 0 else 0,
                "empty_rate": round(v["empty"] / v["present"], 4) if v["present"] > 0 else 0,
                "null_str_count": v["null_str"],
            }
            for k, v in field_quality.items()
        },
        "duplicate_id_field": id_field,
        "duplicate_ids": duplicate_ids,
        "duplicate_rate": round(duplicate_rate, 4),
        "encoding_issues": encoding_issues,
        "html_entities": html_entities,
    }
    print(f"  {cat_info['label']}: dup={duplicate_rate:.2%}, encoding={encoding_issues}, html={html_entities}")



품질 분석:   9%|▉         | 1/11 [00:06<01:09,  6.90s/it]

  판례: dup=0.00%, encoding=0, html=0


품질 분석:  18%|█▊        | 2/11 [00:08<00:33,  3.73s/it]

  법령: dup=0.00%, encoding=0, html=0


품질 분석:  27%|██▋       | 3/11 [00:10<00:22,  2.87s/it]

...

  법률용어사전: dup=2.21%, encoding=0, html=1


품질 분석:  91%|█████████ | 10/11 [00:27<00:01,  1.43s/it]

  조약: dup=0.00%, encoding=0, html=6


품질 분석: 100%|██████████| 11/11 [00:28<00:00,  2.55s/it]

  행정규칙: dup=0.02%, encoding=0, html=0


In [3]:
# ── null률 히트맵 (카테고리 x 필드) ──────────────────────
# 주요 필드: text_fields + id_field + date_field + summary_field
key_fields_per_cat = {}
for cat_key, cat_info in CATEGORIES.items():
    fields = set()
    if cat_info.get("id_field"):
        fields.add(cat_info["id_field"])
    if cat_info.get("date_field"):
        fields.add(cat_info["date_field"])
    for tf in cat_info.get("text_fields", []):
        fields.add(tf)
    if cat_info.get("summary_field"):
        fields.add(cat_info["summary_field"])
    key_fields_per_cat[cat_key] = fields

# 모든 주요 필드의 합집합
all_key_fields = sorted(set().union(*key_fields_per_cat.values()))

# 히트맵 데이터 구성
heatmap_data = []
cat_labels = []
for cat_key, qr in quality_results.items():
    cat_labels.append(qr["label"])
    row = []
    for field in all_key_fields:
        fq = qr["field_quality"].get(field)
        if fq is None:
            row.append(None)  # 해당 카테고리에 이 필드 없음
        else:
            row.append(fq["null_rate"] + fq["empty_rate"])  # null + empty 합산
    heatmap_data.append(row)

fig = px.imshow(
    heatmap_data,
    x=all_key_fields,
    y=cat_labels,
    title="카테고리 x 주요 필드 null/empty 비율 히트맵<br><sub>text_fields + summary_field + id/date 필드 포함</sub>",
    labels=dict(x="필드", y="카테고리", color="null+empty 비율"),
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig.update_layout(height=500)
fig.show()

In [4]:
# ── 중복 ID 비율 바 차트 ─────────────────────────────────
dup_data = [
    {
        "카테고리": qr["label"],
        "중복 비율": qr["duplicate_rate"],
        "중복 ID 수": qr["duplicate_ids"],
        "ID 필드": qr["duplicate_id_field"] or "없음",
    }
    for qr in quality_results.values()
]
df_dup = pd.DataFrame(dup_data).sort_values("중복 비율", ascending=True)

fig = px.bar(
    df_dup,
    x="중복 비율",
    y="카테고리",
    orientation="h",
    title="카테고리별 ID 중복 비율",
    hover_data=["중복 ID 수", "ID 필드"],
    color="중복 비율",
    color_continuous_scale="Reds",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 2. 텍스트 길이 분석

주요 텍스트 필드의 길이 분포를 분석하여 청킹 전략을 도출합니다.

In [5]:
# ── 카테고리별 텍스트 길이 수집 (원본 + 요약 필드) ─────────
text_stats = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="텍스트 분석"):
    text_fields = list(cat_info.get("text_fields", []))
    summary_field = cat_info.get("summary_field")

    if summary_field:
        text_fields.append(summary_field)

    if not text_fields or cat_key not in _available_categories:
        continue

    field_lengths: dict[str, list[int]] = {f: [] for f in text_fields}

    for record in iter_category_records(cat_key):
        for field in text_fields:
            value = record.get(field)
            if isinstance(value, str) and value.strip():
                field_lengths[field].append(len(value))
            elif isinstance(value, list):
                field_lengths[field].append(len(str(value)))

    cat_stats = {}
    for field, lengths in field_lengths.items():
        if not lengths:
            continue
        arr = np.array(lengths)
        is_summary = (field == summary_field)
        cat_stats[field] = {
            "count": len(lengths),
            "min": int(arr.min()),
            "p25": int(np.percentile(arr, 25)),
            "p50": int(np.percentile(arr, 50)),
            "p75": int(np.percentile(arr, 75)),
            "p90": int(np.percentile(arr, 90)),
            "p99": int(np.percentile(arr, 99)),
            "max": int(arr.max()),
            "mean": round(float(arr.mean()), 1),
            "is_summary": is_summary,
        }

    text_stats[cat_key] = {
        "label": cat_info["label"],
        "fields": cat_stats,
    }
    for field, stats in cat_stats.items():
        tag = " [요약]" if stats["is_summary"] else ""
        print(f"  {cat_info['label']}.{field}{tag}: P50={stats['p50']:,}, P90={stats['p90']:,}, max={stats['max']:,}")



텍스트 분석:   9%|▉         | 1/11 [00:05<00:53,  5.36s/it]

  판례.판례내용: P50=89, P90=141, max=2,712
  판례.판결요지: P50=369, P90=899, max=12,370
  판례.판시사항: P50=102, P90=283, max=2,712
  판례.이유: P50=2,270, P90=7,562, max=861,517
  판례.판례요약 [요약]: P50=230, P90=321, max=831


텍스트 분석:  18%|█▊        | 2/11 [00:07<00:31,  3.49s/it]

  법령.조문: P50=11,752, P90=48,949, max=747,801
  법령.법령 요약 [요약]: P50=720, P90=1,288, max=2,203


텍스트 분석:  27%|██▋       | 3/11 [00:08<00:20,  2.56s/it]

...

  법률용어사전.법령용어정의: P50=62, P90=286, max=102,778


텍스트 분석:  91%|█████████ | 10/11 [00:23<00:01,  1.20s/it]

  조약.조약내용: P50=2,730, P90=14,609, max=400,747
  조약.조약요약 [요약]: P50=232, P90=292, max=473


텍스트 분석: 100%|██████████| 11/11 [00:23<00:00,  2.17s/it]

  행정규칙.조문내용: P50=2,248, P90=7,904, max=64,681
  행정규칙.행정규칙요약 [요약]: P50=532, P90=699, max=1,759


In [6]:
# ── 카테고리별 텍스트 길이 box plot ──────────────────────
# 원본 필드와 요약 필드를 색상으로 구분
box_data = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        tag = " [요약]" if stats.get("is_summary") else ""
        box_data.append({
            "카테고리": ts["label"],
            "필드": field,
            "레이블": f"{ts['label']}\n{field}{tag}",
            "P25": stats["p25"],
            "P50": stats["p50"],
            "P75": stats["p75"],
            "P90": stats["p90"],
            "P99": stats["p99"],
            "mean": stats["mean"],
            "유형": "요약" if stats.get("is_summary") else "원본",
        })

df_box = pd.DataFrame(box_data)

# 원본/요약 색상 매핑
color_map = {"원본": "#636EFA", "요약": "#EF553B"}

fig = go.Figure()
for _, row in df_box.iterrows():
    color = color_map[row["유형"]]
    fig.add_trace(go.Box(
        name=row["레이블"],
        q1=[row["P25"]],
        median=[row["P50"]],
        q3=[row["P75"]],
        lowerfence=[row["P25"]],
        upperfence=[row["P90"]],
        mean=[row["mean"]],
        boxmean=True,
        marker_color=color,
        line_color=color,
    ))

fig.update_layout(
    title="카테고리별 텍스트 길이 분포 (P25-P90, 문자 수)<br><sub>🔵 원본 필드  🔴 요약 필드</sub>",
    yaxis_title="텍스트 길이 (문자)",
    height=700,
    showlegend=False,
)
fig.show()

In [7]:
# ── 요약 필드 텍스트 길이 히스토그램 (전체 카테고리) ──────
from plotly.subplots import make_subplots

CHUNK_SIZE = 1250  # 현재 청킹 기준선

summary_cats = []
for cat_key, cat_info in CATEGORIES.items():
    summary_field = cat_info.get("summary_field")
    if not summary_field or cat_key not in _available_categories:
        continue

    lengths = []
    for r in iter_category_records(cat_key):
        v = r.get(summary_field)
        if isinstance(v, str) and v.strip():
            lengths.append(len(v))

    if lengths:
        summary_cats.append({
            "cat_key": cat_key,
            "label": cat_info["label"],
            "summary_field": summary_field,
            "lengths": lengths,
        })

n = len(summary_cats)
cols = 2
rows = (n + 1) // cols

fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[
        f"{sc['label']} — {sc['summary_field']} ({len(sc['lengths']):,}건)"
        for sc in summary_cats
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

for i, sc in enumerate(summary_cats):
    r = i // cols + 1
    c = i % cols + 1
    fig.add_trace(
        go.Histogram(
            x=sc["lengths"],
            nbinsx=80,
            marker_color="#EF553B",
            name=sc["label"],
            showlegend=False,
        ),
        row=r, col=c,
    )
    fig.add_vline(
        x=CHUNK_SIZE, line_dash="dash", line_color="red",
        row=r, col=c,
    )

fig.update_layout(
    title=f"카테고리별 요약 필드 텍스트 길이 분포 (빨간 점선: 청킹 기준 {CHUNK_SIZE}자)",
    height=300 * rows,
)
fig.update_xaxes(title_text="문자 수")
fig.update_yaxes(title_text="빈도")
fig.show()



In [8]:
# ── 예상 청크 수 분석 (원본 + 요약 필드) ─────────────────
# 현재 청킹 설정: 1,250자 / 800토큰 기준
CHUNK_CHAR_LIMIT = 1250
OVERLAP_CHARS = 200  # 가정: 200자 오버랩

chunking_analysis = []
for cat_key, ts in text_stats.items():
    for field, stats in ts["fields"].items():
        # 평균 텍스트 길이 기준 예상 청크 수
        effective_chunk = CHUNK_CHAR_LIMIT - OVERLAP_CHARS
        avg_chunks = max(1, stats["mean"] / effective_chunk) if effective_chunk > 0 else 1
        p90_chunks = max(1, stats["p90"] / effective_chunk) if effective_chunk > 0 else 1

        field_type = "요약" if stats.get("is_summary") else "원본"
        chunking_analysis.append({
            "카테고리": ts["label"],
            "필드": field,
            "유형": field_type,
            "평균 길이": stats["mean"],
            "P90 길이": stats["p90"],
            "평균 청크 수": round(avg_chunks, 1),
            "P90 청크 수": round(p90_chunks, 1),
            "청킹 필요": "예" if stats["p50"] > CHUNK_CHAR_LIMIT else "아니오",
        })

df_chunk = pd.DataFrame(chunking_analysis)

# 유형별 색상 적용
fill_colors = []
for col in df_chunk.columns:
    col_colors = []
    for _, row in df_chunk.iterrows():
        if row["유형"] == "요약":
            col_colors.append("#FFF3F3")  # 연한 빨강
        else:
            col_colors.append("#F2F2F2")
    fill_colors.append(col_colors)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_chunk.columns),
        fill_color="#548235",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_chunk[col] for col in df_chunk.columns],
        fill_color=fill_colors,
        align="left",
        font=dict(size=11),
        height=28,
    ),
)])
fig.update_layout(
    title=f"청킹 전략 권고 (기준: {CHUNK_CHAR_LIMIT}자, 오버랩: {OVERLAP_CHARS}자)<br><sub>⬜ 원본 필드  🟥 요약 필드</sub>",
    height=max(400, len(chunking_analysis) * 30 + 100),
)
fig.show()

# ── 요약 필드 vs 원본 필드 청킹 비교 ──────────────────────
print("\n=== 요약 필드 vs 원본 필드 청킹 비교 ===")
for cat_key, ts in text_stats.items():
    orig_fields = {f: s for f, s in ts["fields"].items() if not s.get("is_summary")}
    summary_fields = {f: s for f, s in ts["fields"].items() if s.get("is_summary")}
    if not summary_fields:
        continue

    # 원본 필드 중 가장 긴 것
    if orig_fields:
        longest_orig = max(orig_fields.items(), key=lambda x: x[1]["p50"])
        orig_name, orig_stats = longest_orig
    else:
        continue

    for sum_name, sum_stats in summary_fields.items():
        ratio = sum_stats["p50"] / orig_stats["p50"] if orig_stats["p50"] > 0 else 0
        needs_chunk_orig = "Y" if orig_stats["p50"] > CHUNK_CHAR_LIMIT else "N"
        needs_chunk_sum = "Y" if sum_stats["p50"] > CHUNK_CHAR_LIMIT else "N"
        print(f"  {ts['label']}: {orig_name}(P50={orig_stats['p50']:,}, 청킹={needs_chunk_orig}) → {sum_name}(P50={sum_stats['p50']:,}, 청킹={needs_chunk_sum}) | 요약 비율={ratio:.1%}")


=== 요약 필드 vs 원본 필드 청킹 비교 ===
  판례: 이유(P50=2,270, 청킹=Y) → 판례요약(P50=230, 청킹=N) | 요약 비율=10.1%
  법령: 조문(P50=11,752, 청킹=Y) → 법령 요약(P50=720, 청킹=N) | 요약 비율=6.1%
  헌재결정례: 이유(P50=678, 청킹=N) → 심판례요약(P50=289, 청킹=N) | 요약 비율=42.6%
  행정심판례: 이유(P50=3,382, 청킹=Y) → 심판례요약(P50=241, 청킹=N) | 요약 비율=7.1%
  특별행정심판: 이유(P50=4,265, 청킹=Y) → 심판례요약(P50=234, 청킹=N) | 요약 비율=5.5%
  법령해석례: 이유(P50=2,557, 청킹=Y) → 해석례요약(P50=200, 청킹=N) | 요약 비율=7.8%
  위원회 결정문: 이유(P50=3,948, 청킹=Y) → 결정문요약(P50=194, 청킹=N) | 요약 비율=4.9%
  부처 해석례: 회답(P50=320, 청킹=N) → 해석요약(P50=193, 청킹=N) | 요약 비율=60.3%
  조약: 조약내용(P50=2,730, 청킹=Y) → 조약요약(P50=232, 청킹=N) | 요약 비율=8.5%
  행정규칙: 조문내용(P50=2,248, 청킹=Y) → 행정규칙요약(P50=532, 청킹=N) | 요약 비율=23.7%


In [9]:
# ── 결과 저장 ─────────────────────────────────────────────
p3_path = save_result("phase3_quality", quality_results)
print(f"Phase 3 저장: {p3_path}")

p4_path = save_result("phase4_text", text_stats)
print(f"Phase 4 저장: {p4_path}")

# 요약 출력
print(f"\n=== 품질 요약 ===")
for cat_key, qr in quality_results.items():
    issues = []
    if qr["duplicate_rate"] > 0:
        issues.append(f"ID중복 {qr['duplicate_rate']:.1%}")
    if qr["encoding_issues"] > 0:
        issues.append(f"인코딩 {qr['encoding_issues']}건")
    if qr["html_entities"] > 0:
        issues.append(f"HTML {qr['html_entities']}건")
    issue_str = ", ".join(issues) if issues else "이슈 없음"
    print(f"  {qr['label']}: {issue_str}")

Phase 3 저장: /Users/gimjuhyeong/dev/law-3-team/backend/eda_output/phase3_quality.json
Phase 4 저장: /Users/gimjuhyeong/dev/law-3-team/backend/eda_output/phase4_text.json

=== 품질 요약 ===
  판례: 이슈 없음
  법령: 이슈 없음
  헌재결정례: HTML 3건
  행정심판례: HTML 4건
  특별행정심판: ID중복 8.6%
  법령해석례: 이슈 없음
  위원회 결정문: ID중복 20.6%
  부처 해석례: 이슈 없음
  법률용어사전: ID중복 2.2%, HTML 1건
  조약: HTML 6건
  행정규칙: ID중복 0.0%
